In [1]:
import pandas as pd
import numpy as np

## Known and Unsolved Cases
I tested the flags against two lists I put together from news and court sources. One has 50 confirmed and caught serial killers. The other has 11 unsolved but suspected serial killer activity, which is what this method is really built to find. Every case has at least 3 women killed in one county, with one main weapon, in one decade.

In [2]:
unknown = pd.read_csv("../data/sk_unknown.csv")
known = pd.read_csv("../data/sk.csv")

## Grouping by Decade
Serial killers are usually active for a few years, not 44, so I rebuilt the groups by decade. Otherwise a few bad years get buried in decades of normal ones.

In [3]:
df = pd.read_csv("../data/scored.csv")
df["Weapon"] = df["Weapon"].replace({"Strangulation - hanging": "Strangulation", "Asphyxiation - includes death by gas": "Strangulation"})
df["Decade"] = (df["Year"] // 10) * 10
df["n_in_incident"] = df.groupby("ID")["ID"].transform("size")
df["variance"] = df["n_in_incident"] * df["p_solved"] * (1 - df["p_solved"])
df["unsolved_inc"] = (1 - df["solved"]) / df["n_in_incident"]

keys = ["CNTYFIPS", "VicSex", "Weapon", "Decade"]
decade = df.groupby(keys)[["solved", "p_solved", "variance", "unsolved_inc"]].sum()
decade["n_murders"] = df.groupby(keys)["solved"].count()
decade = decade.reset_index()
decade = decade.rename(columns={"solved": "actual_solved", "p_solved": "expected_solved"})
decade["n_unsolved"] = decade["n_murders"] - decade["actual_solved"]
decade["z_score"] = (decade["actual_solved"] - decade["expected_solved"]) / np.sqrt(decade["variance"])

county_dec = decade.groupby(["CNTYFIPS", "Decade"])[["actual_solved", "expected_solved"]].sum().reset_index()
county_dec = county_dec.rename(columns={"actual_solved": "county_actual", "expected_solved": "county_expected"})
decade = decade.merge(county_dec, on=["CNTYFIPS", "Decade"], how="left")

decade["county_ratio"] = (decade["county_actual"] - decade["actual_solved"]) / (decade["county_expected"] - decade["expected_solved"])
decade["county_ratio"] = decade["county_ratio"].replace([np.inf, -np.inf], np.nan)
decade["adj_expected"] = decade["expected_solved"] * decade["county_ratio"]
decade["adj_z_score"] = (decade["actual_solved"] - decade["adj_expected"]) / np.sqrt(decade["variance"])

print(decade.shape)

(92425, 16)


92,425 groups.

In [4]:
fem_dec = decade[(decade["VicSex"] == "Female") & (decade["unsolved_inc"] >= 5)].copy()
fem_dec.shape

(1459, 16)

1,459 female-victim groups with at least 5 separate unsolved incidents.

## Flagging Groups
I flagged any group below −3, and I picked that cutoff before checking either list. A score that low almost never happens by chance.

In [5]:
flagged = fem_dec[fem_dec["adj_z_score"] < -3].sort_values("adj_z_score")
print(len(flagged))
flagged[["CNTYFIPS", "Weapon", "Decade", "n_murders", "n_unsolved", "adj_z_score"]]

37


,CNTYFIPS,Weapon,Decade,n_murders,n_unsolved,adj_z_score
67152,"Polk, FL","Personal weapons, includes beating",2000,9,5,-5.903924
6393,"Bexar, TX",Knife or cutting instrument,1990,51,24,-5.014708
21206,"Dauphin, PA","Personal weapons, includes beating",1980,11,7,-4.585445
77404,"Spokane, WA","Handgun - pistol, revolver, etc",1990,26,11,-4.535716
5312,"Bell, TX","Handgun - pistol, revolver, etc",1970,9,6,-4.398141
21776,"De Kalb, GA",Rifle,2010,6,5,-4.126173
60030,"New York, NY","Personal weapons, includes beating",2000,155,66,-4.041597
6134,"Bernalillo, NM",Other or type unknown,2000,14,13,-3.959136
20809,"Dallas, TX","Firearm, type not stated",1990,57,48,-3.803483
39690,"Jackson, MO",Strangulation,1980,42,34,-3.745797


37 of the 1,459 groups got flagged.

## Confirmed Killers

In [6]:
check = known.merge(fem_dec, on=["CNTYFIPS", "Weapon", "Decade"], how="left")
check["Scoreable"] = np.where(check["adj_z_score"].notna(), "Yes", "No")
check = check.sort_values("adj_z_score")
check["flagged"] = np.where(check["adj_z_score"] < -3, "Yes", "No")
print("Scoreable:", (check["Scoreable"] == "Yes").sum(), "of", len(check))
check[check["Scoreable"] == "Yes"][["Killer", "CNTYFIPS", "Weapon", "Decade", "n_unsolved", "n_murders", "adj_z_score", "flagged"]].style.hide(axis="index").format(precision=2)

Scoreable: 34 of 50


Killer,CNTYFIPS,Weapon,Decade,n_unsolved,n_murders,adj_z_score,flagged
Robert Lee Yates,"Spokane, WA","Handgun - pistol, revolver, etc",1990,11.00,26.00,-4.54,Yes
Lorenzo Gilyard,"Jackson, MO",Strangulation,1980,34.00,42.00,-3.75,Yes
Dellmus Colvin,"Lucas, OH",Strangulation,2000,5.00,6.00,-3.30,Yes
Derrick Todd Lee,"East Baton Rouge, LA",Knife or cutting instrument,2000,7.00,18.00,-2.60,No
Gerald Parker,"Orange, CA","Blunt object - hammer, club, etc",1970,10.00,14.00,-2.16,No
Sean Vincent Gillis,"East Baton Rouge, LA",Strangulation,2000,7.00,10.00,-2.00,No
Benjamin Atkins,"Wayne, MI",Strangulation,1990,58.00,89.00,-1.82,No
Dayton Leroy Rogers,"Clackamas, OR",Knife or cutting instrument,1980,9.00,13.00,-1.60,No
Coral Eugene Watts,"Harris, TX",Strangulation,1980,32.00,52.00,-1.28,No
Harrison Graham,"Philadelphia, PA",Strangulation,1980,32.00,55.00,-1.15,No


34 of the 50 killers fall in a group the model can score. Yates (Spokane), Gilyard (Kansas City), and Colvin (Toledo) got flagged. Derrick Todd Lee just missed at −2.60.

## Unsolved Cases

In [7]:
unsolved_check = unknown.merge(fem_dec, on=["CNTYFIPS", "Weapon", "Decade"], how="left")
unsolved_check["Scoreable"] = np.where(unsolved_check["adj_z_score"].notna(), "Yes", "No")
unsolved_check = unsolved_check.sort_values("adj_z_score")
unsolved_check["flagged"] = np.where(unsolved_check["adj_z_score"] < -3, "Yes", "No")
unsolved_check[["Killer", "Status", "CNTYFIPS", "Weapon", "Decade", "n_unsolved", "n_murders", "adj_z_score", "flagged"]].style.hide(axis="index").format(precision=2)

Killer,Status,CNTYFIPS,Weapon,Decade,n_unsolved,n_murders,adj_z_score,flagged
West Mesa murders,Unsolved - treated as serial,"Bernalillo, NM",Other or type unknown,2000,13.00,14.00,-3.96,Yes
New Bedford Highway Killer,Unsolved - treated as serial,"Bristol, MA",Other or type unknown,1980,6.00,7.00,-1.91,No
South Memphis sex worker killings,Unsolved - police linked by DNA,"Shelby, TN","Firearm, type not stated",2010,19.00,52.00,-1.68,No
Texas Killing Fields (Calder Road),Unsolved - prime suspect died; 2026 indictment in 2 cases,"Galveston, TX",Other or type unknown,1980,5.00,5.00,-1.31,No
Golden Years Murders (stabbings),Unsolved - first six victims stabbed,"Richmond, VA",Knife or cutting instrument,1990,20.00,31.00,-1.09,No
Storyville Slayer,Unsolved - all but one case open,"Orleans, LA",Strangulation,1990,6.00,8.00,-0.89,No
Honolulu Strangler,Unsolved - police suspected serial killer,"Honolulu, HI",Strangulation,1980,7.00,10.00,-0.66,No
Frankford Slasher,Unsolved - treated as serial,"Philadelphia, PA",Knife or cutting instrument,1980,39.00,158.00,-0.25,No
San Diego serial murders,Unsolved - Metropolitan Homicide Task Force series; some later solved,"San Diego, CA",Other or type unknown,1980,10.00,18.00,0.36,No
Eastbound Strangler,Unsolved - treated as serial,"Atlantic, NJ",Strangulation,2000,nan,nan,nan,No


9 of the 11 unsolved series could be scored. The West Mesa murders got flagged. That case is still open, so the model picked out a real unsolved series without being told about it.

## How Often the Flags Hit
To see if the hits are more than luck, I compared how often groups with a known case got flagged to how often any group gets flagged. Some killers share a group, so I counted each group once.

In [8]:
base_rate = (fem_dec["adj_z_score"] < -3).mean()

def summarize(df, label):
    s = df[df["Scoreable"] == "Yes"].drop_duplicates(["CNTYFIPS", "Weapon", "Decade"])
    rate = (s["flagged"] == "Yes").mean()
    print(label)
    print("  Flagged, all cases:", (df["flagged"] == "Yes").sum(), "of", len(df))
    print("  Flagged, unique scoreable groups:", (s["flagged"] == "Yes").sum(), "of", len(s))
    print("  Flag rate:", round(rate * 100, 1), "% | baseline:", round(base_rate * 100, 1), "% | lift:", round(rate / base_rate, 1))

summarize(check, "All confirmed killers")
summarize(check[check["Added"] == "Nov 2026"], "Clean set (added without seeing scores)")
summarize(unsolved_check, "Unsolved series")

All confirmed killers
  Flagged, all cases: 3 of 50
  Flagged, unique scoreable groups: 3 of 31
  Flag rate: 9.7 % | baseline: 2.5 % | lift: 3.8
Clean set (added without seeing scores)
  Flagged, all cases: 1 of 27
  Flagged, unique scoreable groups: 1 of 10
  Flag rate: 10.0 % | baseline: 2.5 % | lift: 3.9
Unsolved series
  Flagged, all cases: 1 of 11
  Flagged, unique scoreable groups: 1 of 9
  Flag rate: 11.1 % | baseline: 2.5 % | lift: 4.4


Groups with a confirmed killer got flagged 9.7% of the time, compared to 2.5% for all groups, so about 3.8 times more often. The killers I added later did about the same (1 of 10 groups). The unsolved series came in at 11.1%, about 4.4 times more often.

In [9]:
fem_dec.to_csv("../data/fem_dec.csv", index=False)
flagged.to_csv("../data/flagged.csv", index=False)